In [ ]:
!pip -q install transformers accelerate bitsandbytes


In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from PIL import Image

from transformers import AutoProcessor

# Try the official LLaVA class; fallback if your transformers version differs
try:
    from transformers import LlavaForConditionalGeneration
    MODEL_CLASS = "llava"
except Exception:
    from transformers import AutoModelForVision2Seq
    MODEL_CLASS = "auto_v2s"


def safe_apply_chat_template(processor, user_text: str):
    """
    Uses a model's chat template if available (recommended for LLaVA).
    Falls back to a simple format if not supported.
    """
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "user", "content": [{"type": "text", "text": user_text}]}
        ]
        return processor.apply_chat_template(conversation, add_generation_prompt=True)
    else:
        # fallback (less ideal, but works for some models)
        return f"USER: {user_text}\nASSISTANT:"


def analyze_generation(out, processor, prompt_len: int):
    """
    out: generate() output with return_dict_in_generate=True, output_scores=True
    prompt_len: number of tokens in the prompt (to slice generated tokens)
    Returns a DataFrame with per-token logprob/entropy + top alternatives.
    """
    # Generated token ids (exclude prompt tokens)
    gen_ids = out.sequences[:, prompt_len:]
    # out.scores is a list of logits tensors, one per generated step
    steps = len(out.scores)

    rows = []
    for t in range(steps):
        step_logits = out.scores[t]              # [1, vocab]
        step_logp = F.log_softmax(step_logits, dim=-1)  # [1, vocab]
        step_p = step_logp.exp()

        token_id = gen_ids[0, t].item()
        token_logprob = step_logp[0, token_id].item()
        token_prob = float(torch.exp(torch.tensor(token_logprob)))
        entropy = (-(step_p * step_logp).sum(dim=-1)[0]).item()

        # Decode current generated token (may include whitespace markers)
        token_str = processor.tokenizer.decode([token_id], skip_special_tokens=False)

        # Top-5 alternatives (use logprobs for stability)
        topk = torch.topk(step_logp[0], k=5)
        top_ids = topk.indices.tolist()
        top_lps = topk.values.tolist()
        top_tokens = [processor.tokenizer.decode([i], skip_special_tokens=False) for i in top_ids]
        top_probs = [float(torch.exp(torch.tensor(lp))) for lp in top_lps]

        rows.append({
            "t": t,
            "token": token_str,
            "logprob": token_logprob,
            "prob": token_prob,
            "entropy": entropy,
            "top5_tokens": top_tokens,
            "top5_probs": top_probs
        })

    df = pd.DataFrame(rows)

    # Summary scores
    avg_logprob = df["logprob"].mean() if len(df) else float("nan")
    avg_entropy = df["entropy"].mean() if len(df) else float("nan")
    min_logprob = df["logprob"].min() if len(df) else float("nan")
    max_entropy = df["entropy"].max() if len(df) else float("nan")

    # A simple "internal confidence" score (NOT calibrated correctness!)
    # This is the geometric mean token probability.
    internal_conf = float(torch.exp(torch.tensor(avg_logprob))) if len(df) else float("nan")

    return df, {
        "avg_logprob": float(avg_logprob),
        "avg_entropy": float(avg_entropy),
        "min_logprob": float(min_logprob),
        "max_entropy": float(max_entropy),
        "internal_conf": internal_conf
    }

In [ ]:
from transformers import BitsAndBytesConfig

model_id = "llava-hf/llava-1.5-7b-hf"  # ✅ recommended for Colab T4

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

processor = AutoProcessor.from_pretrained(model_id)

if MODEL_CLASS == "llava":
    model = LlavaForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    ).eval()
else:
    model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    ).eval()

print("Loaded:", model_id)

In [ ]:
from google.colab import files
uploaded = files.upload()

# pick the first uploaded file
img_path = next(iter(uploaded.keys()))
image = Image.open(img_path).convert("RGB")
image

In [ ]:
# Put <image> in the user message for LLaVA-style models
user_text = (
    "<image>\n"
    "Question: What is the main object in the image?\n"
    "Answer with ONE noun only."
)

prompt_text = safe_apply_chat_template(processor, user_text)

inputs = processor(text=prompt_text, images=image, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,                 # ✅ deterministic for analysis
        output_scores=True,              # ✅ gives logits per generated token
        return_dict_in_generate=True
    )

# Decode final answer text (generated part only)
prompt_len = inputs["input_ids"].shape[1]
gen_ids = out.sequences[:, prompt_len:]
answer = processor.tokenizer.decode(gen_ids[0], skip_special_tokens=True)

df, summary = analyze_generation(out, processor, prompt_len)

print("MODEL ANSWER:", answer)
print("\nSUMMARY:")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
# Show tokens sorted by highest entropy (most uncertain)
df.sort_values("entropy", ascending=False).head(10)

In [ ]:
top_uncertain = df.sort_values("entropy", ascending=False).head(5)

for _, row in top_uncertain.iterrows():
    print(f"\nStep {int(row['t'])} token={repr(row['token'])}  entropy={row['entropy']:.3f}")
    for tok, p in zip(row["top5_tokens"], row["top5_probs"]):
        print(f"   {repr(tok)}  p={p:.4f}")